In [1]:
import os
import glob
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
import timm
from facenet_pytorch import MTCNN
from tqdm.notebook import tqdm

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}. Sang Monster Bangkit!")

Menggunakan device: cuda:0. Sang Monster Bangkit!


In [2]:
class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']

# --- 1. PERSIAPAN TRANSFORMASI ---
transform_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [3]:
# --- 2. LOAD THE MONSTER ---
print("Memuat ConvNeXt-V2 Base (Ultimate)...")
model = timm.create_model('convnextv2_base.fcmae_ft_in22k_in1k', pretrained=False, num_classes=len(class_names))
model.load_state_dict(torch.load('../models/convnextv2_base_ultimate.pth', map_location=device))
model = model.to(device).eval()

Memuat ConvNeXt-V2 Base (Ultimate)...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20988\3012616189.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('../models/convnextv2_base

In [4]:
# --- 3. MULTI-ZOOM MTCNN ---
mtcnn_tight = MTCNN(keep_all=False, select_largest=True, margin=20, post_process=False, device=device)
mtcnn_normal = MTCNN(keep_all=False, select_largest=True, margin=60, post_process=False, device=device)
mtcnn_wide = MTCNN(keep_all=False, select_largest=True, margin=100, post_process=False, device=device)

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_pat

In [5]:
# --- 4. PROSES INFERENCE ENDGAME ---
test_dir = '../Data/test/' 
submission_df = pd.read_csv('../Data/samplesubmission.csv')
predictions = []

print("Memulai Prediksi Mutlak dengan Monster ConvNeXt-V2...")
with torch.no_grad():
    for img_id in tqdm(submission_df['id']):
        search_pattern = os.path.join(test_dir, f"{img_id}.*")
        matching_files = glob.glob(search_pattern)
        
        if not matching_files:
            predictions.append('fake_unknown')
            continue
            
        img_path = matching_files[0] 
        
        try:
            img = Image.open(img_path).convert('RGB')
            faces = [mtcnn_tight(img), mtcnn_normal(img), mtcnn_wide(img)]
            
            all_probs = []
            
            for i, face in enumerate(faces):
                if face is not None:
                    pil_img = transforms.ToPILImage()(face.byte())
                else:
                    # Smart Center Crop
                    w, h = img.size
                    ratio = 0.4 if i == 0 else (0.6 if i == 1 else 0.8)
                    margin = (1.0 - ratio) / 2.0
                    left, top = w * margin, h * margin
                    right, bottom = w * (1.0 - margin), h * (1.0 - margin)
                    pil_img = img.crop((left, top, right, bottom))
                    
                tensor_input = transform_224(pil_img).unsqueeze(0).to(device)
                
                # Tebak dengan Monster
                prob = F.softmax(model(tensor_input), dim=1)
                all_probs.append(prob)
            
            # Rata-rata dari 3 level zoom
            avg_prob = torch.mean(torch.stack(all_probs), dim=0)
            
            # Ambil kelas pemenang
            _, predicted_idx = torch.max(avg_prob, 1)
            predicted_class = class_names[predicted_idx.item()]
            
            predictions.append(predicted_class)
            
        except Exception as e:
            print(f"Error memproses {img_path}: {e}")
            predictions.append('fake_unknown')

Memulai Prediksi Mutlak dengan Monster ConvNeXt-V2...


  0%|          | 0/404 [00:00<?, ?it/s]

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [6]:
# --- 5. SIMPAN CSV ---
submission_dir = '../Data/submission/'
os.makedirs(submission_dir, exist_ok=True)
submission_df['label'] = predictions

file_name = 'submission_findit_monster_v2_ultimate.csv'
submission_file_path = os.path.join(submission_dir, file_name)
submission_df.to_csv(submission_file_path, index=False)

print("="*40)
print(f"File submission Monster Ultimate berhasil disimpan!")
print(f"Lokasi: {submission_file_path}")
print("="*40)

File submission Monster Ultimate berhasil disimpan!
Lokasi: ../Data/submission/submission_findit_monster_v2_ultimate.csv
